# 9.19 — Advanced Reasoning (Self-Consistency, Tree/Graph of Thoughts)

Advanced reasoning wraps a base model in a small control system: sample several reasoning traces, vote over their answers, search a tree of partial thoughts, reuse shared subclaims in a graph, or update a belief after an observation. In this notebook we make those ideas concrete with tiny pure-Python/NumPy reasoners so the formula $\hat y=\arg\max_y\sum_{r\in\mathcal{R}(y)}p(r,y\mid x)$ becomes arithmetic you can inspect.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build advanced reasoning one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is visible, including vote counts, branch scores, graph sharing, and odds updates. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, sampling, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
from collections import Counter, defaultdict, deque  # voting and tiny search utilities.
np.random.seed(0)  # reproducibility for toy sampling.

### 1. Self-consistency: sample traces, then vote on answers

A single chain of thought is one sampled path through many possible intermediate statements. Self-consistency says: do not trust one path too much. Instead, sample several traces, map each trace to a final answer, and choose the answer with the largest total probability mass. In the simplest classroom version every sampled trace has equal weight, so the formula becomes a majority vote.

In [ ]:
traces_w = ["split 6 as 3+3 -> A", "pair odds -> A", "misread one case -> B", "count directly -> A", "shortcut -> B"]
answers_w = np.array([t.split("-> ")[1] for t in traces_w])
counts_w = Counter(answers_w)
print("answers:", answers_w.tolist())
print("vote counts:", dict(counts_w))

▶ What you'll see: five sampled traces end in answers `[A, A, B, A, B]`, so A appears three times.

In [ ]:
majority_answer_w, majority_count_w = counts_w.most_common(1)[0]
majority_frac_w = majority_count_w / len(answers_w)
print("winner:", majority_answer_w)
print("majority fraction:", round(majority_frac_w, 3))
assert majority_answer_w == "A" and round(majority_frac_w, 3) == 0.600

▶ What you'll see: A wins with `3/5 = 0.600`, matching the lesson's worked mechanic.

In [ ]:
plt.figure(figsize=(4.2, 3))
plt.bar(list(counts_w.keys()), list(counts_w.values()), color=["seagreen", "indianred"])
plt.axhline(len(answers_w) / 2, color="black", linestyle="--", label="half of samples")
plt.title("1: self-consistency vote")
plt.ylabel("sampled traces")
plt.legend()
plt.show()

▶ What you'll see: A's bar rises above the halfway reference line, while B stays below it.

*Why it's done this way:* if trace errors are partly independent, wrong answers scatter across alternatives while the stable answer accumulates more mass. Majority vote is an empirical estimate of $\sum_{r\in\mathcal{R}(y)}p(r,y\mid x)$ under equal-probability samples; weighted voting is the same idea when traces have different likelihoods.

### 2. Tree of thoughts: search partial reasoning states

Tree-of-thought methods make the control loop explicit. A node is a partial thought, branching proposes continuations, scoring estimates promise, and the search budget decides how many nodes we can afford. With branching factor $b$ and depth $d$, a full tree has $1+b+b^2+\dots+b^d$ nodes, so even tiny branching grows quickly.

In [ ]:
branching_w = 3
max_depth_w = 2
nodes_by_depth_w = np.array([branching_w ** depth for depth in range(max_depth_w + 1)])
total_nodes_w = int(nodes_by_depth_w.sum())
print("nodes by depth:", nodes_by_depth_w.tolist())
print("total nodes:", total_nodes_w)
assert total_nodes_w == 13

▶ What you'll see: depth counts `[1, 3, 9]` sum to `13`, the lesson's tree-search mechanic.

In [ ]:
root_w = {"text": "start", "score": 0.0, "parent": -1, "depth": 0}
children_w = {
    0: [("try algebra", 0.50), ("try cases", 0.62), ("guess", 0.10)],
    1: [("simplify equation", 0.72), ("expand badly", 0.30), ("check units", 0.55)],
    2: [("case even", 0.68), ("case odd", 0.80), ("case zero", 0.58)],
    3: [("commit guess", 0.20), ("revise", 0.18), ("stop", 0.05)],
}
nodes_w = [root_w]
frontier_w = [0]
print("frontier starts at node:", frontier_w)

▶ What you'll see: search begins with only the root thought.

In [ ]:
for parent_w in list(frontier_w):
    for text_w, score_w in children_w[parent_w]:
        nodes_w.append({"text": text_w, "score": score_w, "parent": parent_w, "depth": nodes_w[parent_w]["depth"] + 1})
frontier_w = [i for i, n in enumerate(nodes_w) if n["depth"] == 1]
print("depth-1 thoughts:", [(i, nodes_w[i]["text"], nodes_w[i]["score"]) for i in frontier_w])

▶ What you'll see: three first-step thoughts are proposed and scored.

In [ ]:
best_parent_w = max(frontier_w, key=lambda i: nodes_w[i]["score"])
for text_w, score_w in children_w[best_parent_w]:
    nodes_w.append({"text": text_w, "score": score_w, "parent": best_parent_w, "depth": 2})
best_leaf_w = max([i for i, n in enumerate(nodes_w) if n["depth"] == 2], key=lambda i: nodes_w[i]["score"])
print("expanded parent:", nodes_w[best_parent_w]["text"])
print("best leaf:", nodes_w[best_leaf_w]["text"], "score", nodes_w[best_leaf_w]["score"])
assert nodes_w[best_leaf_w]["text"] == "case odd"

▶ What you'll see: a best-first search expands the most promising first thought and finds `case odd`.

In [ ]:
plt.figure(figsize=(5.2, 3.4))
for i_w, n_w in enumerate(nodes_w):
    x_w = n_w["depth"]
    same_depth_w = [j for j, m in enumerate(nodes_w) if m["depth"] == n_w["depth"]]
    y_w = same_depth_w.index(i_w)
    n_w["xy"] = (x_w, -y_w)
for i_w, n_w in enumerate(nodes_w[1:], start=1):
    px_w, py_w = nodes_w[n_w["parent"]]["xy"]
    x_w, y_w = n_w["xy"]
    plt.plot([px_w, x_w], [py_w, y_w], color="gray", linewidth=1)
for i_w, n_w in enumerate(nodes_w):
    color_w = "seagreen" if i_w == best_leaf_w else "steelblue"
    plt.scatter(*n_w["xy"], s=260, color=color_w)
    plt.text(n_w["xy"][0] + 0.04, n_w["xy"][1], str(i_w), va="center")
plt.title("2: a searched tree of thoughts")
plt.axis("off")
plt.show()

▶ What you'll see: a small reasoning tree; the green leaf is the best scored path, not simply the first path generated.

*Why it's done this way:* tree search spends compute where a scoring function says the expected payoff is high. The math tradeoff is budget versus coverage: $1+b+\dots+b^d$ nodes can explode, so pruning or best-first expansion is what turns exhaustive reasoning into a controllable algorithm.

### 3. Graph of thoughts: reuse shared subclaims instead of recomputing them

A tree duplicates work whenever two paths need the same intermediate claim. A graph-of-thought view stores each unique subclaim once and lets multiple answer paths point to it. The payoff is not magic accuracy; it is memoization and consistency: if two candidate solutions both depend on the same check, that check should be evaluated once and reused.

In [ ]:
paths_w = {
    "path_A": ["parse problem", "check parity", "compute answer A"],
    "path_B": ["parse problem", "check parity", "compute answer B"],
}
all_checks_w = [c for path_w in paths_w.values() for c in path_w]
unique_checks_w = sorted(set(all_checks_w))
print("tree-style checks:", len(all_checks_w))
print("graph unique checks:", len(unique_checks_w))
print("saved evaluations:", len(all_checks_w) - len(unique_checks_w))

▶ What you'll see: two paths contain six check references but only four unique subclaims.

In [ ]:
mini_paths_w = [["shared", "A"], ["shared", "B"]]
refs_w = sum(len(p_w) for p_w in mini_paths_w)
unique_w = len(set(sum(mini_paths_w, [])))
print("lesson mini references:", refs_w)
print("lesson unique checks:", unique_w)
assert refs_w - 1 == unique_w == 3

▶ What you'll see: the lesson's `4-1=3` unique-check arithmetic in the smallest possible graph.

In [ ]:
claim_scores_w = {"parse problem": 0.98, "check parity": 0.75, "compute answer A": 0.70, "compute answer B": 0.55}
path_scores_w = {name_w: float(np.prod([claim_scores_w[c_w] for c_w in path_w])) for name_w, path_w in paths_w.items()}
print("path scores:", {k: round(v, 3) for k, v in path_scores_w.items()})
print("best graph path:", max(path_scores_w, key=path_scores_w.get))
assert max(path_scores_w, key=path_scores_w.get) == "path_A"

▶ What you'll see: shared claims affect both paths equally, so the differing final subclaim decides the winner.

In [ ]:
pos_w = {"parse problem": (0, 0), "check parity": (1, 0), "compute answer A": (2, .5), "compute answer B": (2, -.5)}
edges_w = [("parse problem", "check parity"), ("check parity", "compute answer A"), ("check parity", "compute answer B")]
plt.figure(figsize=(5.2, 3))
for u_w, v_w in edges_w:
    plt.plot([pos_w[u_w][0], pos_w[v_w][0]], [pos_w[u_w][1], pos_w[v_w][1]], color="gray")
for node_w, (x_w, y_w) in pos_w.items():
    plt.scatter(x_w, y_w, s=420, color="darkorange" if node_w == "check parity" else "steelblue")
    plt.text(x_w, y_w + 0.08, node_w, ha="center", fontsize=8)
plt.title("3: graph of thoughts reuses one subclaim")
plt.axis("off")
plt.show()

▶ What you'll see: two answer branches share the orange `check parity` node instead of duplicating it in two separate trees.

*Why it's done this way:* a graph changes the unit of computation from paths to unique subclaims. Mathematically, shared factors should not be multiplied, scored, or paid for twice; caching them keeps costs lower and makes dependent answers agree about the same premise.

### 4. ReAct-style observation update: reason, act, observe, revise

ReAct adds a small feedback loop: a prior belief from reasoning is updated by an observation from an action or tool. The cleanest toy math is odds form. If the prior probability of an answer is $p$, its odds are $p/(1-p)$. Evidence with likelihood ratio $L$ multiplies the odds; converting back gives the posterior probability.

In [ ]:
prior_p_w = 0.55
likelihood_ratio_w = 3.0
prior_odds_w = prior_p_w / (1 - prior_p_w)
posterior_odds_w = prior_odds_w * likelihood_ratio_w
posterior_p_w = posterior_odds_w / (1 + posterior_odds_w)
print("prior odds:", round(prior_odds_w, 3))
print("posterior odds:", round(posterior_odds_w, 3))
print("posterior probability:", round(posterior_p_w, 3))
assert round(posterior_odds_w, 3) == 3.667 and round(posterior_p_w, 3) == 0.786

▶ What you'll see: confidence moves from `0.55` to `0.786` after evidence with likelihood ratio 3.

In [ ]:
states_w = ["reason prior", "act: query", "observe evidence", "revise belief"]
probs_w = [prior_p_w, prior_p_w, posterior_p_w, posterior_p_w]
print(list(zip(states_w, np.round(probs_w, 3))))

▶ What you'll see: probability changes only after the observation enters the state.

In [ ]:
sample_count_w = 5
tokens_per_trace_w = 100
total_tokens_w = sample_count_w * tokens_per_trace_w
print("token cost:", total_tokens_w)
assert total_tokens_w == 500
plt.figure(figsize=(5, 3))
plt.plot(range(len(probs_w)), probs_w, marker="o", color="purple")
plt.xticks(range(len(states_w)), states_w, rotation=20, ha="right")
plt.ylim(0, 1)
plt.ylabel("belief in answer")
plt.title("4: observation updates state")
plt.tight_layout()
plt.show()

▶ What you'll see: a step upward after observation, plus the explicit `5 × 100 = 500` token cost.

*Why it's done this way:* the loop is only useful if observations are parsed into state. Odds multiplication cleanly separates prior reasoning from external evidence, while the token count reminds us that every extra trace, branch, or tool call is a budgeted decision.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, sampling, vectorized scoring, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for vote bars, search curves, and graph drawings.
from collections import Counter, defaultdict, deque # load standard-library containers for voting and tiny search loops.
np.random.seed(0) # make stochastic toy examples reproducible.

def majority_vote(labels): # return the winning label and its empirical probability mass.
    counts = Counter(labels) # count sampled final answers.
    winner, count = counts.most_common(1)[0] # choose the most frequent answer.
    return winner, count / len(labels), counts # expose winner, fraction, and full counts.

def weighted_vote(labels, weights): # aggregate probability mass by answer label.
    totals = defaultdict(float) # store total trace mass for each final answer.
    for label, weight in zip(labels, weights): # add each trace's weight to its answer.
        totals[label] += float(weight) # convert to float for readable numeric output.
    winner = max(totals, key=totals.get) # choose the answer with largest accumulated mass.
    return winner, dict(totals) # return both the argmax and all masses.

def softmax(logits): # convert scores to probabilities for toy reasoner choices.
    z = np.asarray(logits, dtype=float) # ensure numerical arrays.
    e = np.exp(z - np.max(z)) # subtract max for stable exponentials.
    return e / e.sum() # normalize into probabilities.

def tree_node_count(branching, depth): # count full tree nodes through a chosen depth.
    return int(sum(branching ** d for d in range(depth + 1))) # geometric series 1+b+...+b^d.

def posterior_from_lr(prior, likelihood_ratio): # update belief using odds and likelihood ratio.
    odds = prior / (1 - prior) # convert probability to odds.
    post_odds = odds * likelihood_ratio # multiply by evidence strength.
    return post_odds / (1 + post_odds) # convert odds back to probability.

## 🟢 Basics (warm-up)

### Basic 1 — Count a self-consistency majority

**Goal.** Turn sampled trace answers into a majority decision, because self-consistency estimates answer probability by repeated reasoning. We build it in 2 steps.

In [ ]:
answers_b1 = np.array(["A", "A", "B", "A", "B"]) # store five sampled final answers.
winner_b1, frac_b1, counts_b1 = majority_vote(answers_b1) # compute majority vote and fraction.
print("counts:", dict(counts_b1)) # inspect the empirical answer distribution.
print("winner:", winner_b1, "fraction:", round(frac_b1, 3)) # inspect the chosen answer and its mass.
assert winner_b1 == "A" and round(frac_b1, 3) == 0.600 # verify the worked 3/5 majority.

▶ What you'll see: A receives three of five votes.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact vote chart.
plt.bar(list(counts_b1.keys()), list(counts_b1.values()), color="teal") # draw one bar per answer.
plt.title("Basic 1: majority vote") # title the visualization.
plt.ylabel("trace count") # label the count axis.
plt.show() # display the bar chart.

▶ What you'll see: the A bar is taller than the B bar.

👀 Takeaway: self-consistency replaces one fragile trace with an empirical vote over several traces.

### Basic 2 — Convert vote counts into probabilities

**Goal.** Normalize vote counts, because self-consistency is estimating answer mass rather than only counting strings. We build it in 2 steps.

In [ ]:
answers_b2 = np.array(["red", "blue", "red", "red", "blue", "green"]) # define sampled answers.
counts_b2 = Counter(answers_b2) # count how often each answer appears.
total_b2 = len(answers_b2) # count all traces.
print("total traces:", total_b2) # inspect the denominator.

▶ What you'll see: six sampled traces form the denominator for empirical probabilities.

In [ ]:
probs_b2 = {k: v / total_b2 for k, v in counts_b2.items()} # normalize counts into probabilities.
print("probabilities:", {k: round(v, 3) for k, v in probs_b2.items()}) # inspect estimated masses.
assert round(probs_b2["red"], 3) == 0.500 # verify 3/6.
plt.figure(figsize=(4, 3)) # create a probability bar chart.
plt.bar(list(probs_b2.keys()), list(probs_b2.values()), color="seagreen") # draw empirical probabilities.
plt.ylim(0, 1) # keep probability scale fixed.
plt.title("Basic 2: empirical answer mass") # title the plot.
plt.show() # display the plot.

▶ What you'll see: red has probability 0.5, blue 0.333, and green 0.167.

👀 Takeaway: majority vote is an argmax over empirical answer probabilities.

### Basic 3 — Weight traces by confidence

**Goal.** Let higher-confidence traces contribute more mass, because not all sampled paths need be equally likely. We build it in 2 steps.

In [ ]:
labels_b3 = ["A", "B", "A", "B"] # define final answers from four traces.
weights_b3 = np.array([0.35, 0.20, 0.25, 0.20]) # define trace probabilities that sum to 1.
print("total weight:", round(float(weights_b3.sum()), 3)) # verify the mass is normalized.
assert round(float(weights_b3.sum()), 3) == 1.000 # self-check probability mass.

▶ What you'll see: all trace weights sum to 1.

In [ ]:
winner_b3, totals_b3 = weighted_vote(labels_b3, weights_b3) # add weights by final answer.
print("weighted totals:", {k: round(v, 3) for k, v in totals_b3.items()}) # inspect answer masses.
print("winner:", winner_b3) # inspect the weighted decision.
assert winner_b3 == "A" and round(totals_b3["A"], 3) == 0.600 # verify A has the larger mass.
plt.figure(figsize=(4, 3)) # create a compact weighted-vote chart.
plt.bar(list(totals_b3.keys()), list(totals_b3.values()), color="purple") # draw answer masses.
plt.title("Basic 3: weighted self-consistency") # title the plot.
plt.ylabel("probability mass") # label the y-axis.
plt.show() # display the plot.

▶ What you'll see: A wins even though the labels are tied 2–2, because A has more probability mass.

👀 Takeaway: the core formula sums trace probability, not just trace count.

### Basic 4 — Count tree nodes from branching and depth

**Goal.** Compute the size of a full reasoning tree, because search cost grows geometrically with depth. We build it in 2 steps.

In [ ]:
branching_b4 = 3 # each thought proposes three continuations.
depth_b4 = 2 # search two continuation levels after the root.
levels_b4 = np.array([branching_b4 ** d for d in range(depth_b4 + 1)]) # count nodes by depth.
print("nodes by level:", levels_b4.tolist()) # inspect the geometric series pieces.

▶ What you'll see: the tree has 1 root, 3 children, and 9 grandchildren.

In [ ]:
total_b4 = tree_node_count(branching_b4, depth_b4) # sum all levels in the full tree.
print("total nodes:", total_b4) # inspect total search cost.
assert total_b4 == 13 # verify the lesson number 1+3+9.
plt.figure(figsize=(4, 3)) # create a compact node-count plot.
plt.bar(["d0", "d1", "d2"], levels_b4, color="darkorange") # draw count per depth.
plt.title("Basic 4: geometric tree growth") # title the plot.
plt.ylabel("nodes") # label the y-axis.
plt.show() # display the plot.

▶ What you'll see: depth 2 already dominates the node count.

👀 Takeaway: tree-of-thought quality costs tokens because branching multiplies work.

### Basic 5 — Score candidate thoughts with softmax

**Goal.** Convert heuristic scores into probabilities, because a controller often samples or ranks continuations by score. We build it in 2 steps.

In [ ]:
thoughts_b5 = np.array(["algebra", "cases", "guess"]) # candidate next thoughts.
logits_b5 = np.array([1.0, 1.5, -0.2]) # toy controller scores before normalization.
probs_b5 = softmax(logits_b5) # convert scores to probabilities.
print("probabilities:", np.round(probs_b5, 3)) # inspect the distribution.
assert int(np.argmax(probs_b5)) == 1 # verify cases is most likely.

▶ What you'll see: `cases` gets the largest probability because it has the largest logit.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact probability chart.
plt.bar(thoughts_b5, probs_b5, color="steelblue") # draw one bar per continuation.
plt.title("Basic 5: scored continuations") # title the plot.
plt.ylabel("softmax probability") # label the probability axis.
plt.show() # display the plot.

▶ What you'll see: a probability distribution over possible next thoughts.

👀 Takeaway: scoring lets tree search prioritize promising branches instead of expanding blindly.

### Basic 6 — Pick the best leaf in a tiny tree

**Goal.** Choose the highest-scoring final thought, because tree search needs a stopping decision. We build it in 2 steps.

In [ ]:
leaves_b6 = np.array(["A via algebra", "B via cases", "A via check"]) # candidate leaf thoughts.
scores_b6 = np.array([0.62, 0.51, 0.79]) # evaluator scores for the leaves.
best_idx_b6 = int(np.argmax(scores_b6)) # find the highest-scoring leaf.
print("best leaf:", leaves_b6[best_idx_b6], "score:", scores_b6[best_idx_b6]) # inspect the decision.
assert leaves_b6[best_idx_b6] == "A via check" # verify the argmax.

▶ What you'll see: the third leaf wins by score.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact leaf-score plot.
plt.bar(leaves_b6, scores_b6, color=["gray", "gray", "seagreen"]) # highlight the winning leaf.
plt.ylim(0, 1) # keep score scale readable.
plt.title("Basic 6: choose best leaf") # title the plot.
plt.xticks(rotation=15) # rotate labels for readability.
plt.show() # display the plot.

▶ What you'll see: the chosen leaf is the highest bar.

👀 Takeaway: tree search needs an evaluator; without scoring, more branches are just more text.

### Basic 7 — Detect shared subclaims

**Goal.** Count duplicate subclaims, because graph-of-thought methods save work by reusing them. We build it in 2 steps.

In [ ]:
path1_b7 = ["parse", "shared-check", "answer-A"] # first reasoning path.
path2_b7 = ["parse", "shared-check", "answer-B"] # second reasoning path.
all_b7 = path1_b7 + path2_b7 # concatenate tree-style references.
unique_b7 = sorted(set(all_b7)) # keep unique graph nodes.
print("references:", len(all_b7), "unique:", len(unique_b7)) # inspect duplicate savings.

▶ What you'll see: six references collapse to four unique subclaims.

In [ ]:
saved_b7 = len(all_b7) - len(unique_b7) # compute saved evaluations.
print("saved evaluations:", saved_b7) # inspect graph reuse benefit.
assert saved_b7 == 2 # parse and shared-check were each duplicated once.
plt.figure(figsize=(4, 3)) # create a compact reuse chart.
plt.bar(["tree refs", "graph nodes"], [len(all_b7), len(unique_b7)], color=["red", "teal"]) # compare duplicated versus unique work.
plt.title("Basic 7: shared subclaim reuse") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the graph representation has fewer nodes than the tree references.

👀 Takeaway: graph reasoning saves compute when paths share premises.

### Basic 8 — Update a prior with evidence odds

**Goal.** Apply one ReAct-style observation update, because external evidence should change state numerically. We build it in 2 steps.

In [ ]:
prior_b8 = 0.55 # belief before observing evidence.
lr_b8 = 3.0 # likelihood ratio supplied by the observation.
odds_b8 = prior_b8 / (1 - prior_b8) # convert prior probability to odds.
print("prior odds:", round(odds_b8, 3)) # inspect the odds before evidence.

▶ What you'll see: prior probability 0.55 corresponds to odds about 1.222.

In [ ]:
post_b8 = posterior_from_lr(prior_b8, lr_b8) # update odds and convert back to probability.
print("posterior:", round(post_b8, 3)) # inspect the updated belief.
assert round(post_b8, 3) == 0.786 # verify the worked ReAct update.
plt.figure(figsize=(4, 3)) # create a before-after belief chart.
plt.bar(["prior", "posterior"], [prior_b8, post_b8], color=["gray", "seagreen"]) # compare beliefs.
plt.ylim(0, 1) # keep probability scale fixed.
plt.title("Basic 8: evidence update") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the observation raises confidence from 0.55 to about 0.786.

👀 Takeaway: observations matter only after the controller folds them back into state.

### Basic 9 — Compute reasoning token cost

**Goal.** Multiply number of traces by tokens per trace, because reliability tricks spend inference budget. We build it in 2 steps.

In [ ]:
traces_b9 = 5 # number of sampled reasoning traces.
tokens_each_b9 = 100 # generated tokens per trace.
costs_b9 = np.full(traces_b9, tokens_each_b9) # store per-trace costs for inspection.
print("per-trace costs:", costs_b9.tolist()) # inspect the repeated cost.

▶ What you'll see: each of five traces costs 100 tokens.

In [ ]:
total_b9 = int(costs_b9.sum()) # sum token costs across traces.
print("total tokens:", total_b9) # inspect total inference cost.
assert total_b9 == 500 # verify the lesson cost number.
plt.figure(figsize=(4, 3)) # create a compact cost chart.
plt.bar(range(1, traces_b9 + 1), costs_b9, color="slateblue") # draw one bar per trace.
plt.title("Basic 9: sampling cost") # title the plot.
plt.xlabel("trace") # label x-axis.
plt.ylabel("tokens") # label y-axis.
plt.show() # display the chart.

▶ What you'll see: five equal bars sum to 500 tokens.

👀 Takeaway: better reasoning procedures must be judged against their extra compute cost.

### Basic 10 — Stop when confidence is high enough

**Goal.** Add a simple termination rule, because reasoning loops need a stopping condition. We build it in 2 steps.

In [ ]:
beliefs_b10 = np.array([0.52, 0.61, 0.74, 0.83]) # confidence after successive reasoning steps.
threshold_b10 = 0.80 # chosen stopping threshold.
stop_mask_b10 = beliefs_b10 >= threshold_b10 # mark steps that are confident enough.
print("stop mask:", stop_mask_b10.astype(int).tolist()) # inspect where stopping becomes allowed.

▶ What you'll see: only the final step crosses the threshold.

In [ ]:
stop_step_b10 = int(np.argmax(stop_mask_b10)) if np.any(stop_mask_b10) else -1 # find first acceptable stop.
print("first stop step:", stop_step_b10) # inspect the termination point.
assert stop_step_b10 == 3 # verify the fourth belief is the first above threshold.
plt.figure(figsize=(4, 3)) # create a stopping plot.
plt.plot(beliefs_b10, marker="o", color="purple") # draw confidence over steps.
plt.axhline(threshold_b10, color="red", linestyle="--", label="threshold") # mark stopping rule.
plt.title("Basic 10: termination criterion") # title the plot.
plt.legend() # show threshold label.
plt.show() # display the plot.

▶ What you'll see: the curve crosses the red line at the last point.

👀 Takeaway: advanced reasoning needs explicit termination, not just better intermediate thoughts.

## 🟡 Easy

### Easy 1 — Simulate self-consistency on arithmetic traces

**Goal.** Sample noisy toy reasoners and vote, because self-consistency helps when errors are not perfectly correlated. We build it in 3 steps.

In [ ]:
true_answer_e1 = 12 # the toy task is 7 + 5.
trace_outputs_e1 = np.array([12, 11, 12, 13, 12, 12, 10]) # sampled answers from noisy traces.
print("sampled outputs:", trace_outputs_e1.tolist()) # inspect trace diversity.

▶ What you'll see: most traces find 12, but several make different mistakes.

In [ ]:
winner_e1, frac_e1, counts_e1 = majority_vote(trace_outputs_e1) # vote over final answers.
print("counts:", dict(counts_e1)) # inspect empirical masses.
print("winner:", winner_e1, "fraction:", round(frac_e1, 3)) # inspect selected answer.
assert winner_e1 == true_answer_e1 and round(frac_e1, 3) == 0.571 # verify 4/7.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact answer-frequency plot.
plt.bar([str(k) for k in counts_e1.keys()], list(counts_e1.values()), color="seagreen") # draw frequencies.
plt.title("Easy 1: noisy arithmetic traces") # title the plot.
plt.ylabel("votes") # label y-axis.
plt.show() # display the vote plot.

▶ What you'll see: the correct answer's errors are outvoted by the repeated correct result.

👀 Takeaway: self-consistency works best when wrong traces disagree with each other.

### Easy 2 — Run a two-level best-first tree search

**Goal.** Expand only the most promising branch first, because tree search budgets force prioritization. We build it in 3 steps.

In [ ]:
root_children_e2 = [("factor", 0.70), ("brute force", 0.55), ("guess", 0.10)] # first-level thoughts and scores.
best_parent_e2 = max(root_children_e2, key=lambda x: x[1]) # choose branch to expand.
print("expanded branch:", best_parent_e2) # inspect best first-level thought.

▶ What you'll see: `factor` is expanded because it has the highest score.

In [ ]:
second_level_e2 = [("factor -> cancel", 0.80), ("factor -> sign error", 0.25), ("factor -> verify", 0.88)] # children under the best branch.
leaves_e2 = root_children_e2[1:] + second_level_e2 # frontier contains unexpanded siblings plus new children.
best_leaf_e2 = max(leaves_e2, key=lambda x: x[1]) # choose best current leaf.
print("best current leaf:", best_leaf_e2) # inspect the best thought after one expansion.
assert best_leaf_e2[0] == "factor -> verify" # verify search found the strongest leaf.

In [ ]:
names_e2 = [x[0] for x in leaves_e2] # collect leaf names for plotting.
scores_e2 = [x[1] for x in leaves_e2] # collect leaf scores for plotting.
plt.figure(figsize=(5, 3)) # create a compact frontier plot.
plt.bar(range(len(scores_e2)), scores_e2, color=["gray", "gray", "steelblue", "gray", "seagreen"]) # highlight best leaf.
plt.xticks(range(len(scores_e2)), names_e2, rotation=25, ha="right") # label leaves.
plt.ylim(0, 1) # keep score scale fixed.
plt.title("Easy 2: best-first frontier") # title the plot.
plt.tight_layout() # keep labels visible.
plt.show() # display the plot.

▶ What you'll see: the expanded branch produces the highest-scoring current leaf.

👀 Takeaway: tree search turns reasoning into propose–score–expand decisions.

### Easy 3 — Compare tree duplication with graph reuse

**Goal.** Measure saved checks, because graph-of-thought reasoning can reuse common premises across paths. We build it in 3 steps.

In [ ]:
candidate_paths_e3 = {
    "A": ["read", "derive constraint", "compute A"],
    "B": ["read", "derive constraint", "compute B"],
    "C": ["read", "try shortcut", "compute C"],
} # define three candidate reasoning paths.
refs_e3 = sum(len(p) for p in candidate_paths_e3.values()) # count tree-style references.
unique_e3 = len(set(x for p in candidate_paths_e3.values() for x in p)) # count unique graph nodes.
print("references:", refs_e3, "unique nodes:", unique_e3) # inspect work before and after sharing.

▶ What you'll see: repeated `read` and `derive constraint` inflate the tree reference count.

In [ ]:
saved_e3 = refs_e3 - unique_e3 # compute evaluations saved by graph caching.
print("saved checks:", saved_e3) # inspect reuse benefit.
assert saved_e3 == 3 # verify repeated checks saved three evaluations.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact graph-reuse comparison.
plt.bar(["tree", "graph"], [refs_e3, unique_e3], color=["indianred", "seagreen"]) # compare work counts.
plt.title("Easy 3: graph reuse lowers checks") # title the plot.
plt.ylabel("evaluations") # label y-axis.
plt.show() # display the chart.

▶ What you'll see: the graph bar is lower because shared subclaims are evaluated once.

👀 Takeaway: graph-of-thought methods are useful when different paths depend on the same intermediate facts.

### Easy 4 — Combine reasoning prior with observation likelihood

**Goal.** Update two candidate answers after evidence, because ReAct-style loops should let observations change rankings. We build it in 3 steps.

In [ ]:
answers_e4 = np.array(["A", "B"]) # two candidate answers.
priors_e4 = np.array([0.55, 0.45]) # prior belief from reasoning before observation.
lrs_e4 = np.array([3.0, 1.0]) # observation is three times more likely under A than under B.
print("priors:", priors_e4) # inspect starting beliefs.

▶ What you'll see: A starts only slightly ahead of B.

In [ ]:
odds_e4 = priors_e4 / (1 - priors_e4) # convert each marginal probability to odds for demonstration.
post_a_e4 = posterior_from_lr(priors_e4[0], lrs_e4[0]) # update answer A with the evidence likelihood ratio.
print("A posterior:", round(post_a_e4, 3)) # inspect the worked update.
assert round(post_a_e4, 3) == 0.786 # verify the lesson posterior.

In [ ]:
plt.figure(figsize=(4, 3)) # create a prior-posterior comparison.
plt.bar(["A prior", "A posterior"], [priors_e4[0], post_a_e4], color=["gray", "purple"]) # compare before and after evidence.
plt.ylim(0, 1) # keep probability scale fixed.
plt.title("Easy 4: observation lifts A") # title the plot.
plt.show() # display the chart.

▶ What you'll see: the evidence makes A much more credible than the prior alone.

👀 Takeaway: ReAct is a state update pattern; the observation must numerically affect the next decision.

### Easy 5 — Track accuracy versus token budget

**Goal.** Compare more traces with higher cost, because reasoning improvements are not free. We build it in 3 steps.

In [ ]:
trace_counts_e5 = np.array([1, 3, 5, 9]) # candidate sample budgets.
tokens_per_trace_e5 = 80 # assumed average tokens per reasoning trace.
accuracy_e5 = np.array([0.58, 0.66, 0.71, 0.73]) # toy validation accuracies from increasing samples.
cost_e5 = trace_counts_e5 * tokens_per_trace_e5 # compute generated-token cost.
print("costs:", cost_e5.tolist()) # inspect token budgets.

▶ What you'll see: cost grows linearly with number of sampled traces.

In [ ]:
marginal_gain_e5 = np.diff(accuracy_e5) / np.diff(cost_e5) # compute extra accuracy per extra token.
print("marginal gain per token:", np.round(marginal_gain_e5, 5)) # inspect diminishing returns.
assert cost_e5[2] == 400 # verify 5 traces at 80 tokens each.

In [ ]:
plt.figure(figsize=(5, 3)) # create an accuracy-cost plot.
plt.plot(cost_e5, accuracy_e5, marker="o", color="teal") # draw validation accuracy against cost.
plt.title("Easy 5: accuracy versus token budget") # title the plot.
plt.xlabel("generated tokens") # label cost axis.
plt.ylabel("toy accuracy") # label accuracy axis.
plt.ylim(0.5, 0.8) # zoom to useful range.
plt.show() # display the curve.

▶ What you'll see: accuracy improves, but the last extra traces help less than the first ones.

👀 Takeaway: advanced reasoning methods should be evaluated on both quality and inference budget.

## 🔴 Advanced

### Advanced 1 — Weighted self-consistency with correlated errors

**Goal.** Show when voting can fail, because self-consistency helps only if traces are not all making the same mistake. We build it in 4 steps.

In [ ]:
independent_a1 = np.array(["A", "A", "B", "A", "C", "A", "B"]) # wrong answers scatter.
correlated_a1 = np.array(["B", "B", "B", "A", "B", "A", "C"]) # wrong answers share one failure mode.
win_i_a1, frac_i_a1, counts_i_a1 = majority_vote(independent_a1) # vote in independent-error case.
win_c_a1, frac_c_a1, counts_c_a1 = majority_vote(correlated_a1) # vote in correlated-error case.
print("independent winner:", win_i_a1, round(frac_i_a1, 3)) # inspect helpful case.
print("correlated winner:", win_c_a1, round(frac_c_a1, 3)) # inspect failure case.

▶ What you'll see: the independent case selects A, while correlated wrong traces select B.

In [ ]:
assert win_i_a1 == "A" and win_c_a1 == "B" # verify the contrast.
labels_a1 = ["independent errors", "correlated errors"] # labels for plotting.
correct_mass_a1 = [counts_i_a1["A"] / len(independent_a1), counts_c_a1["A"] / len(correlated_a1)] # correct-answer masses.
print("correct masses:", np.round(correct_mass_a1, 3)) # inspect how correlation lowers correct mass.

In [ ]:
plt.figure(figsize=(5, 3)) # create a correlation pitfall plot.
plt.bar(labels_a1, correct_mass_a1, color=["seagreen", "indianred"]) # compare correct answer mass.
plt.axhline(0.5, color="black", linestyle="--") # mark majority threshold.
plt.title("Advanced 1: correlated errors break voting") # title the plot.
plt.ylabel("mass on correct answer A") # label y-axis.
plt.xticks(rotation=10) # rotate labels.
plt.show() # display the chart.

▶ What you'll see: correct mass clears half only when errors are diverse.

In [ ]:
weights_a1 = np.array([0.16, 0.15, 0.08, 0.17, 0.07, 0.18, 0.19]) # toy trace probabilities.
weighted_winner_a1, totals_a1 = weighted_vote(independent_a1, weights_a1) # weight traces by confidence.
print("weighted totals:", {k: round(v, 3) for k, v in totals_a1.items()}) # inspect weighted masses.
assert weighted_winner_a1 == "A" # verify weighted voting still chooses A here.

▶ What you'll see: weighting can help, but it does not solve perfectly correlated failure modes by itself.

👀 Takeaway: self-consistency is a variance-reduction trick, not a guarantee against shared bias.

### Advanced 2 — Beam search over a reasoning tree

**Goal.** Keep only the top-k partial thoughts at each depth, because exhaustive tree-of-thought search is too expensive. We build it in 4 steps.

In [ ]:
beam_width_a2 = 2 # keep two partial paths after each expansion.
levels_a2 = [
    [("start", 0.0)],
    [("algebra", 0.6), ("cases", 0.7), ("guess", 0.1)],
] # define root and first-level candidates.
beam_a2 = sorted(levels_a2[1], key=lambda x: x[1], reverse=True)[:beam_width_a2] # keep top two first-level thoughts.
print("beam after depth 1:", beam_a2) # inspect retained candidates.

▶ What you'll see: `cases` and `algebra` survive; `guess` is pruned.

In [ ]:
expansions_a2 = {
    "cases": [("cases/even", 0.78), ("cases/odd", 0.83), ("cases/zero", 0.50)],
    "algebra": [("algebra/simplify", 0.74), ("algebra/check", 0.80), ("algebra/error", 0.20)],
} # define children for retained beam nodes.
candidates_a2 = [] # collect second-level candidates.
for name_a2, parent_score_a2 in beam_a2: # expand each retained branch.
    for child_a2, child_score_a2 in expansions_a2[name_a2]: # loop over children.
        candidates_a2.append((child_a2, 0.5 * parent_score_a2 + 0.5 * child_score_a2)) # average parent and child scores.
print("candidate count:", len(candidates_a2)) # inspect work after pruning.

In [ ]:
beam2_a2 = sorted(candidates_a2, key=lambda x: x[1], reverse=True)[:beam_width_a2] # keep top two leaves.
print("beam after depth 2:", [(n, round(s, 3)) for n, s in beam2_a2]) # inspect final beam.
assert beam2_a2[0][0] == "cases/odd" # verify highest-scoring path.

In [ ]:
plt.figure(figsize=(5, 3)) # create a beam-score chart.
plt.bar([c[0] for c in candidates_a2], [c[1] for c in candidates_a2], color="steelblue") # draw all expanded candidates.
plt.axhline(beam2_a2[-1][1], color="red", linestyle="--", label="beam cutoff") # mark retained cutoff.
plt.xticks(rotation=25, ha="right") # keep labels readable.
plt.title("Advanced 2: beam search cutoff") # title the plot.
plt.legend() # show cutoff label.
plt.tight_layout() # fit labels.
plt.show() # display the chart.

▶ What you'll see: only candidates above the cutoff remain in the beam.

👀 Takeaway: beam search is a compute-control compromise between greedy one-path reasoning and full tree expansion.

### Advanced 3 — Graph search with memoized subclaim evaluation

**Goal.** Evaluate each subclaim once and reuse it across answers, because graph-of-thought systems should avoid duplicated checks. We build it in 4 steps.

In [ ]:
subclaims_a3 = {
    "parse": [],
    "constraint": ["parse"],
    "bound": ["parse"],
    "answer_A": ["constraint", "bound"],
    "answer_B": ["constraint"],
} # define a tiny directed acyclic graph of dependencies.
base_scores_a3 = {"parse": 0.95, "constraint": 0.80, "bound": 0.70, "answer_A": 0.90, "answer_B": 0.50} # local reliability scores.
print("graph nodes:", list(subclaims_a3.keys())) # inspect subclaims.

▶ What you'll see: two answers share the `constraint` subclaim.

In [ ]:
memo_a3 = {} # cache evaluated subclaim scores.
def eval_claim_a3(name): # recursively score a claim using dependencies.
    if name in memo_a3: # reuse cached score if available.
        return memo_a3[name] # return memoized value.
    dep_scores_a3 = [eval_claim_a3(dep) for dep in subclaims_a3[name]] # evaluate dependencies first.
    memo_a3[name] = base_scores_a3[name] * (np.prod(dep_scores_a3) if dep_scores_a3 else 1.0) # multiply local and dependency support.
    return memo_a3[name] # return computed score.
score_A_a3 = eval_claim_a3("answer_A") # score answer A.
score_B_a3 = eval_claim_a3("answer_B") # score answer B, reusing shared cache.
print("memoized scores:", {k: round(v, 3) for k, v in memo_a3.items()}) # inspect every unique computation.

In [ ]:
print("A score:", round(score_A_a3, 3), "B score:", round(score_B_a3, 3)) # compare final answers.
print("unique evaluations:", len(memo_a3)) # inspect cache size.
assert len(memo_a3) == 5 and score_A_a3 > score_B_a3 # verify reuse and ranking.

In [ ]:
pos_a3 = {"parse": (0, 0), "constraint": (1, .4), "bound": (1, -.4), "answer_A": (2, .1), "answer_B": (2, .8)} # manual graph layout.
plt.figure(figsize=(5, 3.2)) # create a graph plot.
for child_a3, deps_a3 in subclaims_a3.items(): # draw dependency edges.
    for dep_a3 in deps_a3: # one edge per dependency.
        plt.plot([pos_a3[dep_a3][0], pos_a3[child_a3][0]], [pos_a3[dep_a3][1], pos_a3[child_a3][1]], color="gray") # draw edge.
for node_a3, xy_a3 in pos_a3.items(): # draw nodes.
    plt.scatter(*xy_a3, s=420, color="seagreen" if node_a3 in ["answer_A", "answer_B"] else "darkorange") # color answers differently.
    plt.text(xy_a3[0], xy_a3[1] + 0.08, node_a3, ha="center", fontsize=8) # label node.
plt.title("Advanced 3: memoized graph of thoughts") # title the graph.
plt.axis("off") # hide axes.
plt.show() # display the plot.

▶ What you'll see: shared dependency nodes feed multiple answer nodes but are evaluated once.

👀 Takeaway: graph search is most valuable when answer paths overlap in their premises or verification steps.

### Advanced 4 — ReAct loop with a deterministic toy tool

**Goal.** Alternate reasoning and observation in a small loop, because a controller must update state from tool results rather than ignore them. We build it in 4 steps.

In [ ]:
def tool_lookup_a4(query): # define a deterministic inline tool with no network or file I/O.
    table_a4 = {"2*7": 14, "3*5": 15, "4*4": 16} # tiny built-in facts.
    return table_a4[query] # return exact observation.
question_a4 = "2*7" # task for the toy reasoner.
prior_answer_a4 = 13 # deliberately wrong prior guess.
print("prior answer:", prior_answer_a4) # inspect starting state.

▶ What you'll see: the initial reasoning state contains an incorrect guess.

In [ ]:
observation_a4 = tool_lookup_a4(question_a4) # act and observe exact result.
state_a4 = {"question": question_a4, "prior": prior_answer_a4, "observation": observation_a4} # store state.
print("observation:", observation_a4) # inspect external evidence.
assert observation_a4 == 14 # verify toy tool result.

In [ ]:
final_answer_a4 = state_a4["observation"] if state_a4["observation"] != state_a4["prior"] else state_a4["prior"] # revise answer from observation.
print("final answer:", final_answer_a4) # inspect revised result.
assert final_answer_a4 == 14 # verify observation changed the answer.

In [ ]:
plt.figure(figsize=(4, 3)) # create a before-after answer plot.
plt.bar(["prior", "observed/final"], [prior_answer_a4, final_answer_a4], color=["red", "seagreen"]) # compare state values.
plt.title("Advanced 4: ReAct correction") # title the plot.
plt.ylabel("numeric answer") # label y-axis.
plt.show() # display the chart.

▶ What you'll see: the final answer follows the observation, not the stale prior guess.

👀 Takeaway: ReAct helps only when the observation is parsed into state and allowed to revise the next action or answer.

### Advanced 5 — Jointly tune samples, tree depth, and graph reuse

**Goal.** Compare reasoning configurations under a budget, because deployed systems choose the best quality-cost tradeoff rather than the largest search. We build it in 4 steps.

In [ ]:
configs_a5 = np.array([
    [1, 0, 0],
    [5, 0, 0],
    [3, 2, 0],
    [3, 2, 1],
    [7, 2, 1],
], dtype=float) # columns: samples, tree depth, graph reuse flag.
tokens_per_sample_a5 = 60 # cost per sampled trace.
tokens_per_node_a5 = 20 # cost per searched tree node.
print("configs columns = samples, depth, reuse") # describe matrix columns.
print(configs_a5.astype(int)) # inspect configurations.

▶ What you'll see: several reasoning designs with different sampling, search, and reuse choices.

In [ ]:
branching_a5 = 2 # binary toy tree.
samples_a5 = configs_a5[:, 0] # extract sample counts.
depths_a5 = configs_a5[:, 1] # extract tree depths.
reuse_a5 = configs_a5[:, 2] # extract graph reuse flag.
node_counts_a5 = np.array([tree_node_count(branching_a5, int(d)) for d in depths_a5]) # compute tree nodes.
raw_cost_a5 = samples_a5 * tokens_per_sample_a5 + node_counts_a5 * tokens_per_node_a5 # compute cost before reuse.
cost_a5 = raw_cost_a5 * (1 - 0.20 * reuse_a5) # graph reuse saves 20% of search-like work in this toy model.
print("costs:", np.round(cost_a5, 1)) # inspect budget use.

In [ ]:
quality_a5 = 0.55 + 0.04 * np.log1p(samples_a5) + 0.03 * depths_a5 + 0.02 * reuse_a5 # toy quality curve with diminishing sample returns.
efficiency_a5 = quality_a5 / cost_a5 # quality per generated token.
best_idx_a5 = int(np.argmax(efficiency_a5)) # choose best quality-cost tradeoff.
print("quality:", np.round(quality_a5, 3)) # inspect quality estimates.
print("best config:", configs_a5[best_idx_a5].astype(int).tolist()) # inspect selected design.
assert configs_a5[best_idx_a5].tolist() == [1.0, 0.0, 0.0] # verify efficiency favors the cheapest config here.

In [ ]:
plt.figure(figsize=(5, 3)) # create a quality-cost scatter plot.
plt.scatter(cost_a5, quality_a5, s=100, c=efficiency_a5, cmap="viridis") # color points by efficiency.
for i_a5, cfg_a5 in enumerate(configs_a5.astype(int)): # label each configuration.
    plt.text(cost_a5[i_a5] + 2, quality_a5[i_a5], f"s{cfg_a5[0]} d{cfg_a5[1]} g{cfg_a5[2]}", fontsize=8) # annotate point.
plt.colorbar(label="quality / token") # add efficiency colorbar.
plt.title("Advanced 5: reasoning quality-cost frontier") # title the plot.
plt.xlabel("token cost") # label x-axis.
plt.ylabel("toy quality") # label y-axis.
plt.show() # display the frontier.

▶ What you'll see: larger searches can improve quality, but the best quality-per-token point may be much smaller.

👀 Takeaway: self-consistency, tree search, and graph reuse are knobs in one budgeted reasoning system.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Advanced reasoning methods improve reliability by exploring, scoring, or grounding multiple thought paths.

Advanced reasoning spends extra samples or tool calls to aggregate thought paths, search branches, share graph checks, and update beliefs from observations. This is symbolic and CPU-only: no LLM calls and no notebook execution. Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 91419
rng = np.random.default_rng(SEED)
random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(x)))


def softmax(values):
    arr = np.asarray(values, dtype=float)
    shifted = arr - np.max(arr)
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalize_rows(matrix):
    arr = np.asarray(matrix, dtype=float)
    return arr / arr.sum(axis=1, keepdims=True)


def kl_divergence(policy, reference):
    policy = np.asarray(policy, dtype=float)
    reference = np.asarray(reference, dtype=float)
    safe_policy = np.clip(policy, 1e-9, 1.0)
    safe_reference = np.clip(reference, 1e-9, 1.0)
    return float(np.sum(safe_policy * (np.log(safe_policy) - np.log(safe_reference))))


def make_f8_ladder(topic):
    if topic == "rlhf":
        return build_rlhf_ladder()
    if topic == "dpo":
        return build_dpo_ladder()
    if topic == "constitutional":
        return build_constitutional_ladder()
    if topic == "icl":
        return build_icl_ladder()
    if topic == "prompting":
        return build_prompting_ladder()
    if topic == "reasoning":
        return build_reasoning_ladder()
    raise ValueError(topic)


def build_rlhf_ladder():
    action_names = ["concise_helpful", "verbose_loophole", "refusal"]
    return [
        {
            "name": "D1 one prompt/action",
            "actions": action_names,
            "reference": np.array([[0.62, 0.25, 0.13]]),
            "policy": np.array([[0.55, 0.32, 0.13]]),
            "reward_model": np.array([[2.0, 1.0, 0.2]]),
            "human_reward": np.array([[2.0, 1.0, 0.2]]),
            "advantages": np.array([[0.5, -0.2, -0.4]]),
            "ratios": np.array([[1.2, 0.8, 1.0]]),
            "beta": 0.10,
        },
        {
            "name": "D2 few-shot preference-policy set",
            "actions": action_names,
            "reference": normalize_rows([[0.55, 0.30, 0.15], [0.50, 0.25, 0.25]]),
            "policy": normalize_rows([[0.62, 0.28, 0.10], [0.58, 0.30, 0.12]]),
            "reward_model": np.array([[2.1, 1.4, 0.2], [1.8, 1.2, 0.5]]),
            "human_reward": np.array([[2.0, 1.0, 0.2], [1.7, 1.0, 0.7]]),
            "advantages": np.array([[0.6, -0.1, -0.5], [0.4, 0.0, -0.3]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.1, 1.0, 0.8]]),
            "beta": 0.12,
        },
        {
            "name": "D3 distractor reward loopholes",
            "actions": action_names,
            "reference": normalize_rows([[0.48, 0.37, 0.15], [0.45, 0.35, 0.20], [0.50, 0.30, 0.20]]),
            "policy": normalize_rows([[0.44, 0.48, 0.08], [0.42, 0.45, 0.13], [0.54, 0.34, 0.12]]),
            "reward_model": np.array([[2.0, 2.6, 0.2], [1.9, 2.4, 0.3], [2.1, 1.8, 0.4]]),
            "human_reward": np.array([[2.0, 1.1, 0.2], [1.8, 1.0, 0.4], [2.0, 1.2, 0.6]]),
            "advantages": np.array([[0.4, 0.7, -0.4], [0.3, 0.6, -0.3], [0.5, 0.2, -0.2]]),
            "ratios": np.array([[1.0, 1.4, 0.7], [0.9, 1.3, 0.8], [1.1, 1.1, 0.8]]),
            "beta": 0.08,
        },
        {
            "name": "D4 real-style instruction/reward set",
            "actions": action_names,
            "reference": normalize_rows([[0.50, 0.30, 0.20], [0.45, 0.25, 0.30], [0.40, 0.35, 0.25], [0.55, 0.25, 0.20]]),
            "policy": normalize_rows([[0.63, 0.27, 0.10], [0.58, 0.27, 0.15], [0.55, 0.33, 0.12], [0.66, 0.23, 0.11]]),
            "reward_model": np.array([[2.2, 1.4, 0.5], [2.0, 1.3, 0.8], [1.9, 1.7, 0.4], [2.3, 1.2, 0.3]]),
            "human_reward": np.array([[2.1, 1.1, 0.5], [1.9, 1.0, 0.9], [1.8, 1.1, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.7, 0.0, -0.4], [0.5, -0.1, -0.2], [0.4, 0.1, -0.4], [0.8, -0.2, -0.5]]),
            "ratios": np.array([[1.2, 0.9, 0.7], [1.2, 1.0, 0.8], [1.1, 1.1, 0.8], [1.2, 0.8, 0.7]]),
            "beta": 0.12,
        },
        {
            "name": "D5 longer context with KL drift",
            "actions": action_names,
            "reference": normalize_rows([[0.52, 0.28, 0.20], [0.49, 0.31, 0.20], [0.46, 0.34, 0.20], [0.50, 0.27, 0.23], [0.47, 0.32, 0.21], [0.51, 0.29, 0.20]]),
            "policy": normalize_rows([[0.40, 0.55, 0.05], [0.42, 0.50, 0.08], [0.44, 0.47, 0.09], [0.61, 0.30, 0.09], [0.46, 0.45, 0.09], [0.63, 0.28, 0.09]]),
            "reward_model": np.array([[2.1, 3.0, 0.4], [2.0, 2.8, 0.4], [1.9, 2.7, 0.3], [2.2, 1.5, 0.5], [2.0, 2.5, 0.4], [2.3, 1.4, 0.3]]),
            "human_reward": np.array([[2.0, 1.0, 0.5], [1.9, 0.9, 0.5], [1.8, 1.0, 0.4], [2.1, 1.1, 0.5], [1.9, 0.8, 0.5], [2.2, 1.0, 0.4]]),
            "advantages": np.array([[0.5, 1.0, -0.3], [0.4, 0.9, -0.3], [0.3, 0.8, -0.4], [0.7, 0.1, -0.4], [0.4, 0.7, -0.3], [0.8, 0.0, -0.5]]),
            "ratios": np.array([[0.9, 1.6, 0.5], [0.9, 1.5, 0.6], [1.0, 1.4, 0.6], [1.2, 1.0, 0.6], [1.0, 1.4, 0.6], [1.2, 0.9, 0.6]]),
            "beta": 0.04,
        },
    ]


def build_dpo_ladder():
    return [
        {
            "name": "D1 one prompt preference",
            "policy_chosen": np.array([-1.0]),
            "policy_rejected": np.array([-2.0]),
            "reference_chosen": np.array([-1.5]),
            "reference_rejected": np.array([-2.2]),
            "labels": np.array([1]),
            "beta": 2.0,
        },
        {
            "name": "D2 few-shot preference set",
            "policy_chosen": np.array([-0.8, -1.1, -1.0]),
            "policy_rejected": np.array([-1.8, -1.7, -2.0]),
            "reference_chosen": np.array([-1.2, -1.4, -1.5]),
            "reference_rejected": np.array([-1.9, -1.9, -2.2]),
            "labels": np.array([1, 1, 1]),
            "beta": 2.0,
        },
        {
            "name": "D3 label noise and distractors",
            "policy_chosen": np.array([-0.7, -1.8, -0.9, -1.3]),
            "policy_rejected": np.array([-1.6, -1.4, -1.7, -1.2]),
            "reference_chosen": np.array([-1.1, -1.5, -1.4, -1.4]),
            "reference_rejected": np.array([-1.8, -1.8, -2.0, -1.6]),
            "labels": np.array([1, 0, 1, 0]),
            "beta": 2.0,
        },
        {
            "name": "D4 real-style pair corpus",
            "policy_chosen": np.array([-0.7, -0.9, -1.1, -0.8, -1.0]),
            "policy_rejected": np.array([-1.8, -1.5, -1.6, -1.7, -1.9]),
            "reference_chosen": np.array([-1.2, -1.2, -1.4, -1.3, -1.5]),
            "reference_rejected": np.array([-1.9, -1.7, -1.9, -1.8, -2.1]),
            "labels": np.array([1, 1, 1, 1, 1]),
            "beta": 1.5,
        },
        {
            "name": "D5 longer context with saturated beta",
            "policy_chosen": np.array([-0.5, -0.6, -2.2, -0.7, -2.0, -0.9]),
            "policy_rejected": np.array([-2.0, -1.9, -1.1, -1.8, -1.0, -1.6]),
            "reference_chosen": np.array([-1.2, -1.3, -1.8, -1.4, -1.7, -1.5]),
            "reference_rejected": np.array([-2.1, -2.0, -1.7, -2.0, -1.6, -1.9]),
            "labels": np.array([1, 1, 0, 1, 0, 1]),
            "beta": 6.0,
        },
    ]


def build_constitutional_ladder():
    principles = ["harmlessness", "honesty", "helpfulness"]
    return [
        {
            "name": "D1 one prompt/principle",
            "principles": principles[:2],
            "weights": np.array([2.0, 1.0]),
            "violations": np.array([[0.3, 0.8]]),
            "revised": np.array([[0.2, 0.2]]),
            "task_success": np.array([0.90]),
        },
        {
            "name": "D2 few-shot critique set",
            "principles": principles,
            "weights": np.array([2.0, 1.0, 1.5]),
            "violations": np.array([[0.4, 0.5, 0.2], [0.2, 0.8, 0.3]]),
            "revised": np.array([[0.2, 0.2, 0.2], [0.2, 0.3, 0.2]]),
            "task_success": np.array([0.88, 0.86]),
        },
        {
            "name": "D3 conflicting principles and distractors",
            "principles": principles,
            "weights": np.array([2.5, 1.0, 0.8]),
            "violations": np.array([[0.6, 0.3, 0.7], [0.5, 0.6, 0.6], [0.2, 0.4, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.5], [0.2, 0.3, 0.5], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.70, 0.72, 0.68]),
        },
        {
            "name": "D4 real-style policy examples",
            "principles": principles,
            "weights": np.array([2.0, 1.3, 1.4]),
            "violations": np.array([[0.5, 0.2, 0.3], [0.2, 0.7, 0.4], [0.4, 0.5, 0.5], [0.3, 0.2, 0.8]]),
            "revised": np.array([[0.2, 0.2, 0.3], [0.2, 0.2, 0.3], [0.2, 0.2, 0.4], [0.2, 0.2, 0.5]]),
            "task_success": np.array([0.86, 0.88, 0.84, 0.82]),
        },
        {
            "name": "D5 longer context with vague scores",
            "principles": principles,
            "weights": np.array([3.0, 1.0, 0.3]),
            "violations": np.array([[0.7, 0.3, 0.8], [0.6, 0.4, 0.9], [0.5, 0.5, 0.8], [0.4, 0.6, 0.9], [0.7, 0.2, 0.7]]),
            "revised": np.array([[0.1, 0.2, 0.7], [0.1, 0.3, 0.8], [0.1, 0.3, 0.7], [0.1, 0.4, 0.8], [0.1, 0.2, 0.6]]),
            "task_success": np.array([0.46, 0.42, 0.48, 0.40, 0.44]),
        },
    ]


def build_icl_ladder():
    return [
        {
            "name": "D1 one prompt",
            "scores": np.array([[2.0, 1.0, 0.0]]),
            "labels": np.array([[1.0, 0.0, 1.0]]),
            "targets": np.array([1]),
            "token_budget": 8,
            "demo_tokens": 6,
            "recency_bonus": 0.5,
        },
        {
            "name": "D2 few-shot set",
            "scores": np.array([[2.2, 1.0, 0.1], [0.4, 1.8, 0.2], [1.7, 0.8, 0.5]]),
            "labels": np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1]),
            "token_budget": 16,
            "demo_tokens": 8,
            "recency_bonus": 0.4,
        },
        {
            "name": "D3 distractors and order flips",
            "scores": np.array([[2.0, 1.9, 0.1, 0.0], [0.2, 1.6, 1.5, 0.1], [1.4, 0.3, 1.3, 0.2], [0.1, 0.2, 1.7, 1.6]]),
            "labels": np.array([[1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 0.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1]),
            "token_budget": 24,
            "demo_tokens": 16,
            "recency_bonus": 0.6,
        },
        {
            "name": "D4 real text-label examples",
            "scores": np.array([[2.3, 1.2, 0.6, 0.1], [0.5, 2.1, 1.0, 0.2], [1.9, 0.8, 0.7, 0.4], [0.4, 1.7, 1.4, 0.3], [2.1, 0.7, 0.8, 0.2]]),
            "labels": np.array([[1.0, 0.0, 1.0, 0.0], [0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [1.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1]),
            "token_budget": 40,
            "demo_tokens": 24,
            "recency_bonus": 0.3,
        },
        {
            "name": "D5 longer context with diluted attention",
            "scores": np.array([[2.0, 1.9, 1.8, 1.7, 0.2, 0.1], [0.3, 1.8, 1.7, 1.6, 0.2, 0.1], [1.7, 1.6, 1.5, 1.4, 0.3, 0.2], [0.2, 1.7, 1.6, 1.5, 0.3, 0.2], [1.8, 1.7, 1.6, 1.5, 0.3, 0.2], [0.2, 1.6, 1.5, 1.4, 0.3, 0.2]]),
            "labels": np.array([[1.0, 0.0, 0.0, 0.0, 1.0, 1.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0, 1.0, 0.0], [0.0, 1.0, 0.0, 0.0, 1.0, 0.0]]),
            "targets": np.array([1, 1, 1, 1, 1, 1]),
            "token_budget": 64,
            "demo_tokens": 54,
            "recency_bonus": 0.7,
        },
    ]


def build_prompting_ladder():
    return [
        {
            "name": "D1 one prompt",
            "base_logits": np.array([[1.0, 0.0]]),
            "demo_boost": 0.8,
            "targets": np.array([0]),
            "prompt_tokens": 1200,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.6,
            "samples": 3,
            "imbalance": 0.0,
        },
        {
            "name": "D2 few-shot set",
            "base_logits": np.array([[0.9, 0.0], [0.2, 0.7], [0.8, 0.1]]),
            "demo_boost": 0.7,
            "targets": np.array([0, 1, 0]),
            "prompt_tokens": 900,
            "window": 2048,
            "format_penalty": -0.5,
            "cot_success": 0.62,
            "samples": 3,
            "imbalance": 0.1,
        },
        {
            "name": "D3 distractors and label imbalance",
            "base_logits": np.array([[0.4, 0.2], [0.1, 0.3], [0.5, 0.3], [0.2, 0.4]]),
            "demo_boost": 0.5,
            "targets": np.array([0, 1, 0, 1]),
            "prompt_tokens": 1500,
            "window": 2048,
            "format_penalty": -1.0,
            "cot_success": 0.55,
            "samples": 5,
            "imbalance": 0.6,
        },
        {
            "name": "D4 real-style QA/instruction corpus",
            "base_logits": np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.2], [0.1, 0.7], [0.8, 0.0]]),
            "demo_boost": 0.6,
            "targets": np.array([0, 1, 0, 1, 0]),
            "prompt_tokens": 1300,
            "window": 4096,
            "format_penalty": -0.4,
            "cot_success": 0.66,
            "samples": 5,
            "imbalance": 0.0,
        },
        {
            "name": "D5 longer context with ungrounded chains",
            "base_logits": np.array([[0.5, 0.4], [0.4, 0.5], [0.6, 0.5], [0.3, 0.4], [0.5, 0.5], [0.4, 0.6]]),
            "demo_boost": 0.3,
            "targets": np.array([0, 1, 0, 1, 0, 1]),
            "prompt_tokens": 3800,
            "window": 4096,
            "format_penalty": -1.2,
            "cot_success": 0.52,
            "samples": 7,
            "imbalance": 0.8,
        },
    ]


def build_reasoning_ladder():
    return [
        {
            "name": "D1 one prompt",
            "votes": ["A", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("path1", "claim_shared"), ("path1", "claim_a"), ("path2", "claim_shared"), ("path2", "claim_b")],
            "prior": 0.55,
            "likelihood_ratio": 3.0,
            "tokens_per_trace": 100,
        },
        {
            "name": "D2 few-shot reasoning set",
            "votes": ["A", "C", "A", "A", "B", "A"],
            "correct": "A",
            "branching": 2,
            "depth": 3,
            "checks": [("p1", "sum"), ("p2", "sum"), ("p3", "unit"), ("p4", "lookup")],
            "prior": 0.60,
            "likelihood_ratio": 2.0,
            "tokens_per_trace": 80,
        },
        {
            "name": "D3 distractor correlated errors",
            "votes": ["B", "B", "A", "B", "A", "B"],
            "correct": "A",
            "branching": 3,
            "depth": 2,
            "checks": [("p1", "bad_hint"), ("p2", "bad_hint"), ("p3", "arithmetic"), ("p4", "bad_hint"), ("p5", "arithmetic")],
            "prior": 0.45,
            "likelihood_ratio": 1.5,
            "tokens_per_trace": 90,
        },
        {
            "name": "D4 real-style arithmetic/tool set",
            "votes": ["A", "A", "A", "C", "A", "B", "A"],
            "correct": "A",
            "branching": 3,
            "depth": 3,
            "checks": [("p1", "calc"), ("p2", "calc"), ("p3", "unit"), ("p4", "calendar"), ("p5", "unit"), ("p6", "lookup")],
            "prior": 0.62,
            "likelihood_ratio": 2.8,
            "tokens_per_trace": 110,
        },
        {
            "name": "D5 longer context with weak scorer",
            "votes": ["B", "B", "B", "A", "A", "B", "C", "B"],
            "correct": "A",
            "branching": 4,
            "depth": 3,
            "checks": [("p1", "misread"), ("p2", "misread"), ("p3", "misread"), ("p4", "calc"), ("p5", "lookup"), ("p6", "misread"), ("p7", "calc"), ("p8", "lookup")],
            "prior": 0.50,
            "likelihood_ratio": 1.2,
            "tokens_per_trace": 140,
        },
    ]


def ladder_frame(ladder):
    rows = []
    for rung in ladder:
        keys = [key for key in rung.keys() if key != "name"]
        size = 0
        for value in rung.values():
            if isinstance(value, np.ndarray):
                size = max(size, int(value.size))
        rows.append({"rung": rung["name"], "size": size, "fields": ", ".join(keys[:5])})
    return pd.DataFrame(rows)


We build a reasoning search summary that aggregates trace votes, counts a finite tree, deduplicates graph subclaims, applies a Bayesian ReAct update, and reports generated-token cost.

In [ ]:

def reasoning_search(votes, branching, depth, checks, prior, likelihood_ratio, tokens_per_trace):
    vote_counts = {answer: votes.count(answer) for answer in sorted(set(votes))}
    majority_answer = max(vote_counts, key=vote_counts.get)
    majority_share = vote_counts[majority_answer] / len(votes)
    tree_nodes = sum(branching ** level for level in range(depth + 1))
    unique_checks = len({claim for path, claim in checks})
    prior_odds = prior / (1.0 - prior)
    posterior_odds = prior_odds * likelihood_ratio
    posterior = posterior_odds / (1.0 + posterior_odds)
    token_cost = len(votes) * tokens_per_trace
    return {
        "vote_counts": vote_counts,
        "majority_answer": majority_answer,
        "majority_share": majority_share,
        "tree_nodes": tree_nodes,
        "unique_checks": unique_checks,
        "posterior": posterior,
        "token_cost": token_cost,
    }


The lesson aggregation is $\hat y=\arg\max_y\sum_{r\in\mathcal{R}(y)}p(r,y\mid x)$. Self-consistency votes over traces; tree and graph search manage branches and shared checks; ReAct updates state from observations.

Votes $[A,A,B,A,B]$ give $A$ a $3/5=0.600$ majority. Branching factor 3 and depth 2 explores $1+3+9=13$ nodes. Two paths sharing one subclaim need $4-1=3$ unique checks. ReAct prior $0.55$ and likelihood ratio 3 gives probability $0.786$. Five traces at 100 tokens cost 500 tokens.

In [ ]:

checks = [("path1", "shared"), ("path1", "claim_a"), ("path2", "shared"), ("path2", "claim_b")]
check = reasoning_search(["A", "A", "B", "A", "B"], 3, 2, checks, 0.55, 3.0, 100)
assert round(check["majority_share"], 3) == 0.600
assert check["tree_nodes"] == 13
assert check["unique_checks"] == 3
assert round(check["posterior"], 3) == 0.786
assert check["token_cost"] == 500
print(check)


## The dataset ladder
Build the F8 D1-D5 ladder inline. Each rung increases context, distractors, policy pressure, or reasoning complexity while staying tiny and CPU-only.

In [ ]:

ladder = make_f8_ladder("reasoning")
preview = ladder_frame(ladder)
print(preview.to_string(index=False))
print("sample rung")
print(ladder[0])


## Run the same method across D1-D5
The same method is applied to every rung; only the rung data changes.

In [ ]:

def evaluate_reasoning_rung(rung, diversify=False):
    votes = list(rung["votes"])
    if diversify and rung["correct"] not in votes[:3]:
        votes[0] = rung["correct"]
    summary = reasoning_search(votes, rung["branching"], rung["depth"], rung["checks"], rung["prior"], rung["likelihood_ratio"], rung["tokens_per_trace"])
    accuracy = float(summary["majority_answer"] == rung["correct"])
    accuracy_per_token = accuracy / summary["token_cost"]
    return {
        "reasoning_accuracy": accuracy,
        "accuracy_per_token": accuracy_per_token,
        "majority_share": summary["majority_share"],
        "tree_nodes": summary["tree_nodes"],
        "unique_checks": summary["unique_checks"],
        "token_cost": summary["token_cost"],
    }


results = []
for rung in ladder:
    row = evaluate_reasoning_rung(rung)
    row["rung"] = rung["name"]
    results.append(row)

results_df = pd.DataFrame(results)
print(results_df[["rung", "reasoning_accuracy", "majority_share", "tree_nodes", "unique_checks", "token_cost"]].to_string(index=False))


## Results visualization
First inspect per-rung artifacts, then a summary curve for the plan metric.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for index, rung in enumerate(ladder):
    counts = {answer: rung["votes"].count(answer) for answer in sorted(set(rung["votes"]))}
    axes[index].bar(counts.keys(), counts.values())
    axes[index].set_title(rung["name"].split()[0])
    axes[index].set_xlabel("answer")
fig.suptitle("Per-rung trace votes")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(results_df["token_cost"], results_df["reasoning_accuracy"], marker="o", label="accuracy")
ax.plot(results_df["token_cost"], results_df["accuracy_per_token"] * 1000.0, marker="s", label="accuracy per 1k tokens")
ax.set_xlabel("generated-token budget")
ax.set_title("Reasoning accuracy versus token budget")
ax.legend()
plt.show()


## Pitfall on the hardest rung
Pitfall on D5: self-consistency helps only when errors are not perfectly correlated. Reproduce correlated wrong votes, then diversify prompts/tools and verify observations.

In [ ]:

d5 = ladder[-1]
correlated = evaluate_reasoning_rung(d5, diversify=False)
diversified = evaluate_reasoning_rung(d5, diversify=True)
verified_checks = len({claim for path, claim in d5["checks"] if claim != "misread"})
print("correlated accuracy", round(correlated["reasoning_accuracy"], 3))
print("diversified accuracy", round(diversified["reasoning_accuracy"], 3))
print("correlated majority share", round(correlated["majority_share"], 3))
print("verified non-misread checks", verified_checks)



## Evaluate it + Practice
- Metric: reasoning accuracy per generated token; compare against a no-skill baseline such as always choosing the reference/default answer.
- Sanity check: D1 must reproduce the exact lesson arithmetic asserted above before you trust D5.
- Ablation: turn off the key idea (KL anchor, reference margin, helpfulness term, retrieved examples, balanced prompt, or diversified traces) and confirm the metric drops or the failure mode appears.
- Failure signals: saturated probabilities, zero token budget, high reward with low human win rate, or improved safety with collapsed helpfulness.
- Keep everything CPU-only and seeded; do not download models or execute training-heavy notebook code.


Practice: Change D5 votes so errors are independent and compare self-consistency.

Practice: Increase tree depth by one and compute the node-cost jump.

Practice: Add a shared graph claim and recompute unique checks.